# Supervised policy training on Colab (stages 02 -> 09)

Branch `perf/batched-mcts` of `TahaSalmani/Chess`.

**Runtime -> Change runtime type -> Hardware accelerator: GPU (T4).**

| step | what | expected |
|---|---|---|
| 1-2 | clone, install deps, fetch 6000 games from Lichess | ~1 min |
| 3 | stage 02 validation + stage 03 transformation -> ~250k positions | ~2 min |
| 4 | stage 07 tensor build + stage 08 base model | ~1 min |
| 5 | stage 09 training, 50 epochs, 10 layers | **~45-60 min** |
| 6 | top-1 accuracy on the held-out test split | ~1 min |
| 7 | optional: stage 10 RL self-play with the weights just trained | ~2 min |

Cell 5 is the long one. Run it and leave the tab open - Colab disconnects an idle session.

The dataset is small on purpose (~250k positions) so the whole thing fits in one session and in Colab's ~13 GB of RAM. See the last cell for how to scale up.

In [ ]:
import os, subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout.strip())
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
cores = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else os.cpu_count()
print('usable vCPU:', cores)
print(subprocess.run(['free', '-g'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'switch the runtime to GPU before continuing'

In [ ]:
!rm -rf /content/Chess
!git clone --branch perf/batched-mcts --single-branch --depth 1 https://github.com/TahaSalmani/Chess.git /content/Chess
%cd /content/Chess
!pip install -q python-chess zstandard scikit-learn tensorboard python-box==6.0.2 ensure pyyaml
!pip install -q -e .
!git log --oneline -1

## 1. Data

`research/fetch_pgn_sample.py` uses an HTTP byte-range request, so it pulls a few MB instead of the 3.5 GB monthly database. First argument is the number of games, second (optional) is how many MB to fetch.

Stage 01 is skipped on purpose: it downloads the whole file.

In [ ]:
!python research/fetch_pgn_sample.py 6000
!ls -la artifacts/Data/

In [ ]:
%%time
!python src/Chess/pipeline/stage_02_DataValidation.py 2>&1 | grep -vE 'Created directory|Config loaded'
!python src/Chess/pipeline/stage_03_DataTransformation.py 2>&1 | grep -vE 'Created directory|Config loaded'
!ls -la artifacts/data_transformation/

In [ ]:
%%time
!python src/Chess/pipeline/stage_07_TorchDataset.py 2>&1 | grep -vE 'Created directory|Config loaded'
!python src/Chess/pipeline/stage_08_TorchBaseModel.py 2>&1 | grep -vE 'Created directory|Config loaded'
!ls -la artifacts/torch_dataset artifacts/torch_base_model
!cat params.yaml

## 2. Stage 09 - training

The Chess logger prints a line every 10 steps, which is ~435 lines per epoch. The grep below keeps only the per-epoch summary, checkpoint saves and early-stopping messages, so you get one line of live progress every ~50 seconds. The unfiltered log is still written to `stage09_full.log`.

`EarlyStopping(patience=5)` may cut the run short if validation loss plateaus - that is intended.

In [ ]:
%%time
!stdbuf -oL python -u src/Chess/pipeline/stage_09_Train_Torch_Model.py 2>&1 \
  | tee stage09_full.log \
  | grep --line-buffered -E 'device|Epoch \[[0-9]+/[0-9]+\] - |Checkpoint saved|EarlyStopping|Early stopping|Final trained'

## 3. Did it actually learn?

Cross-entropy over 4096 classes is hard to read - random guessing is `ln(4096) = 8.32`. Top-1 accuracy on the held-out test split is the number that means something. For reference, a strong policy net on human games lands in the 40-55% top-1 range; anything above ~15% is clearly learning.

In [ ]:
import sys, numpy as np, torch
sys.path.insert(0, '/content/Chess/src')
from Chess.components.torch_base_model import ChessPolicyNet
from Chess.config.configuration import ConfigurationManager

cm = ConfigurationManager()
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ChessPolicyNet(config=cm.get_torch_base_model()).to(dev)
model.load_state_dict(torch.load('artifacts/prepare_torch_callbacks/best_model.pth', map_location=dev))
model.eval()

X = np.load('artifacts/data_transformation/X_test.npy')
y = np.load('artifacts/data_transformation/y_test.npy').astype(np.int64)
print(f'test positions: {len(y)}')

correct = 0
top5 = 0
with torch.no_grad():
    for i in range(0, len(y), 1024):
        xb = torch.from_numpy(X[i:i+1024]).float().to(dev)
        logits = model(xb)
        correct += int((logits.argmax(1).cpu().numpy() == y[i:i+1024]).sum())
        top5 += int((logits.topk(5, dim=1).indices.cpu().numpy() == y[i:i+1024, None]).any(1).sum())

print(f'top-1 accuracy: {correct / len(y) * 100:.2f}%   (random = 0.02%)')
print(f'top-5 accuracy: {top5 / len(y) * 100:.2f}%')

In [ ]:
!ls -la artifacts/prepare_torch_callbacks/
from google.colab import files
files.download('/content/Chess/artifacts/prepare_torch_callbacks/best_model.pth')

## 4. Optional: stage 10 right here

The weights are already on disk, so RL self-play can run in the same session with a network that actually knows something. `POOL_SIZE` in `params.yaml` sets how many games share one batched forward.

With `NUM_GAMES: 10` this takes a couple of minutes on a T4.

In [ ]:
!stdbuf -oL python -u src/Chess/pipeline/stage_10_Pytorch_RL.py 2>&1 \
  | tee stage10_full.log \
  | grep --line-buffered -E 'device|weights|MCTS config|Self-play|round |Game |Epoch \[|saved|completed'

## Scaling up

Everything above is sized for one Colab session. To train on more data:

- more games: `python research/fetch_pgn_sample.py 25000` -> ~1M positions -> stage 09 becomes ~4-5 h, which outlives a free Colab session
- past that, Kaggle is the better host (9 h sessions, 29 GB RAM, 30 free GPU-hours/week), but this account currently has **no DNS at all inside kernels**, so git clone / pip install / PGN download all fail. Kaggle needs phone verification on the account to turn internet on; without it the repo, the PGN and every missing wheel have to be shipped in as an attached Dataset.
- RAM ceiling here is ~13 GB: stage 07 materialises the whole train split as float32, which is ~2.2 GB at 1M positions and ~7 GB at 2.5M. Beyond that, stream from the `.npy` files with a memmap dataset instead of one big tensor.